# Genomic Benchmarks: Human vs Worm Classification using ERNIE-RNA Embeddings

In this notebook, we use the ERNIE-RNA model to extract embeddings for RNA sequences from the `demo_human_or_worm` dataset. We then train a K-Nearest Neighbors (KNN) classifier to distinguish between human and worm sequences.

Since the full dataset contains over 100,000 sequences and extracting embeddings on CPU is computationally expensive, we will sample a subset of the data for this demonstration.

In [1]:
import os
import random
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## 1. Data Preparation

We parse the `.fasta` files and extract the sequences, discarding the headers. We will sample 500 sequences for training and 100 sequences for testing per class. The full dataset consists of 37.5k train seqences per class and 12.5k of test sequences per class

In [2]:
def sample_fasta(input_path, num_samples):
    with open(input_path, 'r') as f:
        lines = f.read().splitlines()
    
    # Fasta format: >header\nsequence
    # We only care about the sequence lines
    sequences = [line for line in lines if not line.startswith('>')]
    
    # Sample sequences
    sampled = random.sample(sequences, min(num_samples, len(sequences)))
    return sampled

data_dir = 'data/genomic_benchmarks/demo_human_or_worm/'
train_human = sample_fasta(os.path.join(data_dir, 'train_human.fasta'), 500)
train_worm = sample_fasta(os.path.join(data_dir, 'train_worm.fasta'), 500)
test_human = sample_fasta(os.path.join(data_dir, 'test_human.fasta'), 100)
test_worm = sample_fasta(os.path.join(data_dir, 'test_worm.fasta'), 100)

print(f"Sampled {len(train_human)} human & {len(train_worm)} worm training sequences.")
print(f"Sampled {len(test_human)} human & {len(test_worm)} worm testing sequences.")

Sampled 500 human & 500 worm training sequences.
Sampled 100 human & 100 worm testing sequences.


The `extract_embedding_forked.py` script expects a simple text file with one sequence per line. Let's write our sampled sequences to text files.

In [3]:
os.makedirs('data/classification_task/splits', exist_ok=True)

def write_seqs(filepath, seqs):
    with open(filepath, 'w') as f:
        for seq in seqs:
            f.write(seq + '\n')

write_seqs('data/classification_task/splits/train_human.txt', train_human)
write_seqs('data/classification_task/splits/train_worm.txt', train_worm)
write_seqs('data/classification_task/splits/test_human.txt', test_human)
write_seqs('data/classification_task/splits/test_worm.txt', test_worm)

print("Sequences written to data/classification_task/splits/ directory.")

Sequences written to data/classification_task/splits/ directory.


## 2. Embedding Extraction

We now use the provided `extract_embedding_forked.py` script to compute the ERNIE-RNA `cls_embedding` for each subset. 
*(Note: extraction on CPU takes ~0.4s per sequence, so this step will take a few minutes.)*

In [4]:
!python extract_embedding_forked.py --seqs_path data/classification_task/splits/train_human.txt --save_path data/classification_task/results/train_human/ --device cpu --layer_idx_emb 11 --skip-attention --skip-all-embedding
!python extract_embedding_forked.py --seqs_path data/classification_task/splits/train_worm.txt --save_path data/classification_task/results/train_worm/ --device cpu --layer_idx_emb 11 --skip-attention --skip-all-embedding
!python extract_embedding_forked.py --seqs_path data/classification_task/splits/test_human.txt --save_path data/classification_task/results/test_human/ --device cpu --layer_idx_emb 11 --skip-attention --skip-all-embedding
!python extract_embedding_forked.py --seqs_path data/classification_task/splits/test_worm.txt --save_path data/classification_task/results/test_worm/ --device cpu --layer_idx_emb 11 --skip-attention --skip-all-embedding

Model Loading Done!!!
Done in 52.731985092163086s!
Model Loading Done!!!
Done in 51.29258465766907s!
Model Loading Done!!!
Done in 11.547994613647461s!
Model Loading Done!!!
Done in 12.577043533325195s!


## 3. KNN Classification

We load the extracted embeddings (`cls_embedding.npy`), assign labels (0 for human, 1 for worm), and train a KNN classifier.

In [5]:
# Load embeddings
# cls_embedding.npy has shape [Batch, 1, 768] because we extracted only layer 11.
X_train_human = np.load('data/classification_task/results/train_human/cls_embedding.npy')[:, 0, :]
X_train_worm = np.load('data/classification_task/results/train_worm/cls_embedding.npy')[:, 0, :]
X_test_human = np.load('data/classification_task/results/test_human/cls_embedding.npy')[:, 0, :]
X_test_worm = np.load('data/classification_task/results/test_worm/cls_embedding.npy')[:, 0, :]

# Create datasets and labels
X_train = np.vstack([X_train_human, X_train_worm])
y_train = np.array([0] * len(X_train_human) + [1] * len(X_train_worm))

X_test = np.vstack([X_test_human, X_test_worm])
y_test = np.array([0] * len(X_test_human) + [1] * len(X_test_worm))

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (1000, 768)
Testing set shape: (200, 768)


In [14]:
# Train KNN Classifier
knn = KNeighborsClassifier(n_neighbors=10, weights='distance')
knn.fit(X_train, y_train)

# Evaluate
y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Human (0)', 'Worm (1)']))

Accuracy: 0.9100

Classification Report:
              precision    recall  f1-score   support

   Human (0)       0.94      0.88      0.91       100
    Worm (1)       0.89      0.94      0.91       100

    accuracy                           0.91       200
   macro avg       0.91      0.91      0.91       200
weighted avg       0.91      0.91      0.91       200



In [ ]:
# from sklearn.linear_model import LogisticRegression
# model = LogisticRegression()
# model.fit(X_train, y_train)